# Redrob Hackathon — Kaggle Dual-GPU Pipeline

**Goal:** Rank top 100 from 100K for Senior AI Engineer.

**Kaggle setup:** *Settings → Accelerator → GPU T4 x2*

### Pipeline
1. Clone + extract dataset
2. Install deps
3. GPU precompute: bi-encoder (DataParallel) + BM25 + cross-encoder + features
4. CPU rank: XGBoost LTR → top 100
5. Validate + download

## Step 1 — Clone + Extract

In [ ]:
!git clone https://github.com/JASHWANTHS07/india-runs.git
%cd india-runs
!apt-get install -qq unrar
!unrar x -o+ dataset/Data.rar dataset/
!ls dataset/India_runs_data_and_ai_challenge/

In [ ]:
import os
CANDIDATES = "dataset/India_runs_data_and_ai_challenge/candidates.jsonl"
size_mb = os.path.getsize(CANDIDATES) / 1e6
print(f"candidates.jsonl: {size_mb:.1f} MB")
assert size_mb > 400
print("Dataset OK")

## Step 2 — Install Dependencies + GPU Check

In [ ]:
!pip install -q sentence-transformers tqdm pandas pyarrow xgboost optuna

import torch
gpu_count = torch.cuda.device_count()
for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_mem / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")
print(f"Total GPUs: {gpu_count}")

## Step 3 — Pre-compute (Dual GPU)

Bi-encoder embeddings with DataParallel + BM25 + cross-encoder re-ranking + 45+ features.

**Expected:** ~15–25 min on 2x T4.

In [1]:
import json, sys, time, os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

In [ ]:
# Ensure we're in the repo directory
if not os.path.exists('src/features.py'):
    os.chdir('india-runs')
print(f'Working dir: {os.getcwd()}')

CANDIDATES = "dataset/India_runs_data_and_ai_challenge/candidates.jsonl"
ARTIFACTS = "./artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)
BATCH_SIZE = 256
CROSS_ENCODER_TOP_N = 1000

# ---- Load candidates & extract features ----
print("Loading candidates + extracting features...")
sys.path.insert(0, '.')
from src.features import extract_features
from src.bm25 import compute_bm25_scores
from src.precompute import JD_QUERY
from dataclasses import asdict

candidates = []
with open(CANDIDATES) as f:
    for line in f:
        candidates.append(json.loads(line))

feat_dicts = []
all_texts = []
for cand in tqdm(candidates, desc="Features"):
    fd = asdict(extract_features(cand))
    feat_dicts.append(fd)
    all_texts.append(fd.get("profile_text", ""))
print(f"  {len(candidates)} candidates loaded")

# ---- BM25 ----
print("Computing BM25 scores...")
t_bm25 = time.time()
bm25_scores = compute_bm25_scores(all_texts, JD_QUERY)
print(f"  BM25 done in {time.time() - t_bm25:.1f}s")

# ---- Bi-encoder embeddings (single GPU, no DataParallel) ----
# DataParallel adds ~5x overhead for SentenceTransformer encoding.
# Single GPU with large batch is much faster (~3-5s/batch vs 25s/batch).
print("Loading bi-encoder...")
model = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda:0")
model.max_seq_length = 256  # profiles rarely exceed 256 tokens
model.half()  # fp16 — 2x faster on T4, negligible quality loss

jd_emb = model.encode([JD_QUERY], normalize_embeddings=True, batch_size=1)
np.save(os.path.join(ARTIFACTS, "jd_embedding.npy"), jd_emb[0])

print(f"Embedding {len(all_texts)} candidates (single GPU, batch={BATCH_SIZE})...")
t0 = time.time()
embeddings = model.encode(
    all_texts,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    device="cuda:0",
)
embeddings = embeddings.astype(np.float32)
np.save(os.path.join(ARTIFACTS, "embeddings.npy"), embeddings)
print(f"Embeddings: {embeddings.shape} in {time.time() - t0:.1f}s")

# Semantic similarities
semantic_scores = embeddings @ jd_emb[0].astype(np.float32)

# ---- Cross-encoder re-ranking top-N ----
print(f"Cross-encoder re-ranking top-{CROSS_ENCODER_TOP_N}...")
del model
torch.cuda.empty_cache()

top_n_idx = np.argsort(-semantic_scores)[:CROSS_ENCODER_TOP_N]
cross_enc = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda:0")
ce_pairs = [(JD_QUERY, all_texts[idx]) for idx in top_n_idx]
ce_raw = cross_enc.predict(ce_pairs, batch_size=128, show_progress_bar=True)

ce_min, ce_max = float(ce_raw.min()), float(ce_raw.max())
ce_range = ce_max - ce_min if ce_max > ce_min else 1.0
cross_encoder_scores = np.zeros(len(all_texts), dtype=np.float32)
for j, idx in enumerate(top_n_idx):
    cross_encoder_scores[idx] = (float(ce_raw[j]) - ce_min) / ce_range

del cross_enc
torch.cuda.empty_cache()

# ---- Save features ----
feat_df = pd.DataFrame(feat_dicts)
feat_df.drop(columns=["profile_text"], inplace=True)
feat_df["bm25_score"] = bm25_scores
feat_df["cross_encoder_score"] = cross_encoder_scores.tolist()
feat_df.to_parquet(os.path.join(ARTIFACTS, "features.parquet"), index=False)
print(f"Features saved: {len(feat_df)} rows, {len(feat_df.columns)} cols")

In [2]:
# Verify
emb = np.load('artifacts/embeddings.npy')
jd = np.load('artifacts/jd_embedding.npy')
df = pd.read_parquet('artifacts/features.parquet')
print(f"embeddings: {emb.shape} | jd: {jd.shape} | features: {len(df)} rows")
print(f"bm25 range: [{df.bm25_score.min():.3f}, {df.bm25_score.max():.3f}]")
print(f"cross-enc range: [{df.cross_encoder_score.min():.3f}, {df.cross_encoder_score.max():.3f}]")
print("Artifacts OK")

embeddings: (100000, 1024) | jd: (1024,) | features: 100000 rows
bm25 range: [0.001, 1.000]
cross-enc range: [0.000, 1.000]
Artifacts OK


## Step 4 — Two-Level XGBoost LTR Ranking (CPU, <3 min)

**Level 1:** Hard filter on `technical_yoe >= 5` (excludes consulting years).  
**Level 2:** XGBoost LTR trains on qualified pool only, domain-focused pseudo-labels.

In [ ]:
!python src/rank.py --artifacts ./artifacts --out ./jashwanth_s.csv --method xgboost --tune --tune-trials 100

## Step 5 — Validate & Download

In [ ]:
import pandas as pd, sys
from dataclasses import fields

df = pd.read_csv('jashwanth_s.csv')
assert len(df) == 100
assert set(df['rank']) == set(range(1, 101))
scores = df.sort_values('rank')['score'].tolist()
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1))

# Tie-break check
rows = df.sort_values('rank')[['rank','score','candidate_id']].values.tolist()
for i in range(len(rows)-1):
    if rows[i][1] == rows[i+1][1]:
        assert rows[i][2] < rows[i+1][2], f"Tie-break fail at ranks {rows[i][0]},{rows[i+1][0]}"

sys.path.insert(0, '.')
from src.features import CandidateFeatures
from src.honeypot import is_honeypot
feat_df = pd.read_parquet('artifacts/features.parquet')
top_feats = feat_df[feat_df['candidate_id'].isin(set(df['candidate_id']))]

# Honeypot check
hp = sum(1 for _, r in top_feats.iterrows()
         if is_honeypot(CandidateFeatures(**{f.name: r[f.name] for f in fields(CandidateFeatures) if f.name in r.index}, profile_text='')))

# Technical YoE check — all top 100 must have >= 5 years non-consulting experience
if 'technical_yoe' in top_feats.columns:
    min_tech_yoe = top_feats['technical_yoe'].min()
    below_5 = (top_feats['technical_yoe'] < 5.0).sum()
    print(f"Technical YoE: min={min_tech_yoe:.1f} | Below 5yr: {below_5}")
    assert below_5 == 0, f"{below_5} candidates with technical_yoe < 5 in top 100!"

print(f"Rows: {len(df)} | Scores: {scores[0]:.4f} → {scores[-1]:.4f} | Honeypots: {hp}")
assert hp == 0
print("All checks passed.")

In [5]:
pd.set_option('display.max_colwidth', 80)
df.sort_values('rank').head(10)[['rank', 'candidate_id', 'score', 'reasoning']]

,rank,candidate_id,score,reasoning
0,1,CAND_0011687,1.0000,Senior NLP Engineer at Niramai with 7+ years building ranking and retrieval ...
1,2,CAND_0020877,1.0000,Applied ML Engineer at CRED with 3+ years building ranking and retrieval sys...
2,3,CAND_0023458,1.0000,AI Research Engineer at BYJU'S with 5 years of applied AI/ML at product comp...
3,4,CAND_0069248,1.0000,AI Research Engineer at Unacademy with 1+ years building ranking and retriev...
4,5,CAND_0081846,1.0000,Lead AI Engineer at Razorpay with 6+ years building ranking and retrieval sy...
5,6,CAND_0022852,0.9999,AI Research Engineer at Zomato with 1+ years building ranking and retrieval ...
6,7,CAND_0030031,0.9999,AI Engineer at Microsoft with 3+ years building ranking and retrieval system...
7,8,CAND_0037160,0.9999,Data Scientist at Haptik with 1+ years building ranking and retrieval system...
8,9,CAND_0042690,0.9999,Senior Software Engineer (ML) at Niramai with 6 years of applied AI/ML at pr...
9,10,CAND_0078508,0.9999,AI Research Engineer at Rephrase.ai with 5 years of applied AI/ML at product...


In [ ]:
!python tests/validate_submission.py jashwanth_s.csv

In [ ]:
!python run_pipeline.py --run_sample sample.json --cpu --out sample_output.csv

In [ ]:
from IPython.display import FileLink
FileLink('jashwanth_s.csv')